# Can search reject a wrong answer as reliably as it confirms a correct one?

Search agents often guess an answer and then search to verify it, so the query contains the guess. This pilot tests, on the development questions of BrowseComp-Plus, whether such a query retrieves documents that merely mention the guess instead of evidence that supports or refutes it.

Outputs of this notebook are committed, so they show only aggregates, `query_id` and `docid`, never text from the dataset.

## 0. Setup

Paths and the seed used throughout, and a check that the downloaded files are in place. Row counts and column names come from the parquet metadata, so no row is read here.

In [1]:
import base64
import hashlib
import json
import re
from pathlib import Path

import bm25s
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DATA = Path("data") if Path("data").exists() else Path("../data")
QUERIES_DIR = DATA / "browsecomp-plus" / "queries" / "data"
CORPUS_DIR = DATA / "browsecomp-plus" / "corpus" / "data"
RESULTS = DATA.parent / "results"
SEED = 0


def describe_parquet(directory: Path) -> None:
    files = sorted(directory.glob("*.parquet"))
    n_rows = sum(pq.ParquetFile(f).metadata.num_rows for f in files)
    columns = pq.ParquetFile(files[0]).schema_arrow.names
    print(f"{directory.parent.name}: {len(files)} files, {n_rows} rows, columns: {columns}")


describe_parquet(QUERIES_DIR)
describe_parquet(CORPUS_DIR)

queries: 6 files, 830 rows, columns: ['query_id', 'query', 'answer', 'gold_docs', 'negative_docs', 'evidence_docs']
corpus: 7 files, 100195 rows, columns: ['docid', 'text', 'url']


## 1. Development and test split

The questions are split at random before any result is computed, so that the choice of questions cannot depend on what we see. All prompts, query templates and thresholds are tuned on the development questions only; the test questions stay untouched until the final run. Only `query_id` is read here, and the split is saved to `results/split.json`.

In [2]:
N_DEV = 100
SPLIT_PATH = RESULTS / "split.json"

query_ids = sorted(
    query_id
    for f in QUERIES_DIR.glob("*.parquet")
    for query_id in pq.read_table(f, columns=["query_id"]).column("query_id").to_pylist()
)
shuffled = np.random.default_rng(SEED).permutation(query_ids).tolist()
split = {"seed": SEED, "dev": sorted(shuffled[:N_DEV]), "test": sorted(shuffled[N_DEV:])}

# The split must stay fixed once saved, so a mismatch stops the notebook instead of replacing it.
if SPLIT_PATH.exists():
    if json.loads(SPLIT_PATH.read_text()) != split:
        raise ValueError(f"{SPLIT_PATH} differs from the split computed here")
    print(f"{SPLIT_PATH} already exists and matches")
else:
    RESULTS.mkdir(exist_ok=True)
    SPLIT_PATH.write_text(json.dumps(split, indent=1) + "\n")
    print(f"Wrote {SPLIT_PATH}")

print(f"Questions: {len(query_ids)} total, {len(set(query_ids))} unique query_id")
print(f"Development: {len(split['dev'])}, test: {len(split['test'])}, "
      f"in both: {len(set(split['dev']) & set(split['test']))}")

../results/split.json already exists and matches
Questions: 830 total, 830 unique query_id
Development: 100, test: 730, in both: 0


## 2. De-obfuscating the development questions

Only the development questions are decrypted, to `data/browsecomp-plus/dev.jsonl`, which git ignores. Each field is base64-decoded and XORed with a repeated SHA-256 digest, as described on the dataset card. To check the result, every document is compared with the corpus. The counts below show how large each question's candidate pool is and whether the gold, evidence and negative lists overlap.

In [3]:
DEV_PATH = DATA / "browsecomp-plus" / "dev.jsonl"
CARD_PATH = QUERIES_DIR.parent / "README.md"
DOC_FIELDS = ["gold_docs", "evidence_docs", "negative_docs"]

# The key comes from the local dataset card so that it never appears in this notebook.
key_digest = hashlib.sha256(CARD_PATH.read_text(encoding="utf-8").strip().splitlines()[-1].encode("utf-8")).digest()


def decrypt_string(ciphertext: str) -> str:
    encrypted = np.frombuffer(base64.b64decode(ciphertext), dtype=np.uint8)
    key = np.resize(np.frombuffer(key_digest, dtype=np.uint8), len(encrypted))
    return (encrypted ^ key).tobytes().decode("utf-8")


def decrypt_row(row: dict) -> dict:
    decrypted = {"query_id": row["query_id"], "query": decrypt_string(row["query"]), "answer": decrypt_string(row["answer"])}
    for field in DOC_FIELDS:
        decrypted[field] = [{k: decrypt_string(v) for k, v in doc.items()} for doc in row[field]]
    return decrypted


dev_table = pq.read_table(QUERIES_DIR, filters=[("query_id", "in", split["dev"])])
dev_rows = sorted((decrypt_row(row) for row in dev_table.to_pylist()), key=lambda row: row["query_id"])

with DEV_PATH.open("w", encoding="utf-8") as f:
    for row in dev_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

dev_docs = [doc for row in dev_rows for field in DOC_FIELDS for doc in row[field]]
dev_docids = {doc["docid"] for doc in dev_docs}
corpus_table = pq.read_table(CORPUS_DIR, columns=["docid", "text"], filters=[("docid", "in", list(dev_docids))])
corpus_text = dict(zip(corpus_table.column("docid").to_pylist(), corpus_table.column("text").to_pylist()))

print(f"Decrypted {len(dev_rows)} development questions, {len(dev_docs)} document entries, {len(dev_docids)} unique docid")
print(f"Unique docid found in the corpus: {sum(d in corpus_text for d in dev_docids)} of {len(dev_docids)}")
print(f"Entries with text identical to the corpus: {sum(corpus_text.get(d['docid']) == d['text'] for d in dev_docs)} of {len(dev_docs)}")

doc_counts = pd.DataFrame([
    {
        "gold": len(gold := {d["docid"] for d in row["gold_docs"]}),
        "evidence": len(evidence := {d["docid"] for d in row["evidence_docs"]}),
        "negative": len(negative := {d["docid"] for d in row["negative_docs"]}),
        "pool": len(gold | evidence | negative),
        "gold_in_evidence": len(gold & evidence),
        "gold_in_negative": len(gold & negative),
        "evidence_in_negative": len(evidence & negative),
    }
    for row in dev_rows
])
print(f"Questions without a gold document: {(doc_counts['gold'] == 0).sum()}")
print(doc_counts.describe().loc[["mean", "min", "50%", "max"]].round(2).to_string())

Decrypted 100 development questions, 8475 document entries, 8173 unique docid
Unique docid found in the corpus: 8173 of 8173
Entries with text identical to the corpus: 8475 of 8475
Questions without a gold document: 0
      gold  evidence  negative    pool  gold_in_evidence  gold_in_negative  evidence_in_negative
mean  2.59      6.19     75.97   82.16              2.59               0.0                   0.0
min   1.00      2.00      1.00   12.00              1.00               0.0                   0.0
50%   2.00      6.00     73.50   79.00              2.00               0.0                   0.0
max   7.00     11.00    167.00  173.00              7.00               0.0                   0.0


## 3. BM25 baseline with the neutral query

BM25 is a classic keyword ranking: a document scores higher when it contains the query's rare words often, with a correction for document length. Here each development question gets its own index over its own candidate pool, and the question itself is the query (q0). Parameters are the `bm25s` defaults, with English stopwords and no stemming, and nothing is tuned.

Pool sizes differ a lot between questions, so each result is shown next to what a random ranking would give for the same question. The intervals are 95% bootstrap intervals over questions.

In [4]:
K_VALUES = [5, 10, 20]
N_BOOT = 2000


def build_pool(row: dict, rng: np.random.Generator) -> list[dict]:
    docs = {doc["docid"]: doc for field in DOC_FIELDS for doc in row[field]}
    # Gold documents are listed first, and BM25 score ties would otherwise favour them.
    return [docs[docid] for docid in rng.permutation(list(docs))]


def rank_pool(query: str, pool: list[dict]) -> list[str]:
    retriever = bm25s.BM25()
    retriever.index(bm25s.tokenize([doc["text"] for doc in pool], stopwords="en", show_progress=False), show_progress=False)
    query_tokens = bm25s.tokenize([query], stopwords="en", return_ids=False, show_progress=False)
    doc_indices, _ = retriever.retrieve(query_tokens, k=len(pool), show_progress=False)
    return [pool[i]["docid"] for i in doc_indices[0]]


def score_ranking(ranking: list[str], gold: set[str]) -> dict:
    n = len(ranking)
    scores = {"pool": n, "first_gold_rank": next(i for i, docid in enumerate(ranking, start=1) if docid in gold)}
    # Under a random ranking the expected first gold rank is (n + 1) / (g + 1).
    scores["random_first_gold_rank"] = (n + 1) / (len(gold) + 1)
    for k in K_VALUES:
        scores[f"recall@{k}"] = len(gold & set(ranking[:k])) / len(gold)
        scores[f"random_recall@{k}"] = min(k, n) / n
    return scores


def bootstrap_mean_ci(values: np.ndarray, n_boot: int = N_BOOT, level: float = 0.95, seed: int = SEED) -> tuple[float, float]:
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), (n_boot, len(values)))].mean(axis=1)
    tail = (1 - level) / 2 * 100
    return tuple(np.percentile(means, [tail, 100 - tail]))


pool_rng = np.random.default_rng(SEED)
q0_scores = pd.DataFrame([
    {"query_id": row["query_id"], **score_ranking(rank_pool(row["query"], build_pool(row, pool_rng)), {d["docid"] for d in row["gold_docs"]})}
    for row in dev_rows
])
q0_scores.to_csv(RESULTS / "bm25_q0_dev.csv", index=False)

summary = []
for metric in [f"recall@{k}" for k in K_VALUES] + ["first_gold_rank"]:
    difference = (q0_scores[metric] - q0_scores[f"random_{metric}"]).to_numpy()
    low, high = bootstrap_mean_ci(q0_scores[metric].to_numpy())
    diff_low, diff_high = bootstrap_mean_ci(difference)
    summary.append({
        "metric": metric, "bm25": q0_scores[metric].mean(), "ci_low": low, "ci_high": high,
        "random": q0_scores[f"random_{metric}"].mean(), "difference": difference.mean(), "diff_ci_low": diff_low, "diff_ci_high": diff_high,
    })

print(f"BM25 with q0 on {len(q0_scores)} development questions; pools of {q0_scores['pool'].min()}-{q0_scores['pool'].max()} documents")
print(f"Median first gold rank: {q0_scores['first_gold_rank'].median()}")
print(f"Pools of at most k documents: " + ", ".join(f"k={k}: {(q0_scores['pool'] <= k).sum()}" for k in K_VALUES))
print(pd.DataFrame(summary).round(3).to_string(index=False))

BM25 with q0 on 100 development questions; pools of 12-173 documents
Median first gold rank: 20.0
Pools of at most k documents: k=5: 0, k=10: 0, k=20: 1
         metric   bm25  ci_low  ci_high  random  difference  diff_ci_low  diff_ci_high
       recall@5  0.060   0.034    0.089   0.072      -0.012       -0.040         0.017
      recall@10  0.148   0.099    0.199   0.144       0.004       -0.041         0.049
      recall@20  0.277   0.215    0.346   0.282      -0.004       -0.063         0.059
first_gold_rank 28.570  24.069   33.261  27.165       1.405       -2.124         5.052
